# picoNewton v3 — mechanosensory observability in anisotropic Womersley flow

This notebook preserves the six-artery `picoNewton_v2` model and tests a dimensionally closed two-state mechanosensor. It reports verified and reproduction paths separately and does not identify a receptor or equate the Lamb force with total wall traction.

Primary references: DOI `10.1038/s41598-026-47474-x`, DOI `10.1073/pnas.0307804101`, DOI `10.1242/jcs.264456`.

In [ ]:
import os
PROFILE=os.environ.get('PICONEWTON_PROFILE','quick')
REPOSITORY_REF=os.environ.get('PICONEWTON_REF','main')
STORAGE_MODE_OVERRIDE=os.environ.get('PICONEWTON_STORAGE')
DEVELOPMENT_SKIP_V2_HASH=os.environ.get('PICONEWTON_DEV_SKIP_V2_HASH','0')=='1'
assert PROFILE in {'quick','publication'}
print({'profile':PROFILE,'repository_ref':REPOSITORY_REF})

## 1. Environment and repository

In [ ]:
from __future__ import annotations
import json, shutil, subprocess, sys
from pathlib import Path

def locate_root():
    for p in [Path.cwd(),*Path.cwd().parents]:
        if p.name=='picoNewton_v3' and (p/'src/piconewton_v3').is_dir(): return p.resolve()
        if (p/'picoNewton_v3/src/piconewton_v3').is_dir(): return (p/'picoNewton_v3').resolve()
PROJECT_ROOT=locate_root()
if PROJECT_ROOT is None and 'google.colab' in sys.modules:
    checkout=Path('/content/picoNewton')
    if not checkout.exists(): subprocess.run(['git','clone','--depth','1','--branch',REPOSITORY_REF,'https://github.com/khalid-saqr/picoNewton.git',str(checkout)],check=True)
    PROJECT_ROOT=checkout/'picoNewton_v3'
if PROJECT_ROOT is None: raise RuntimeError('picoNewton_v3 not found')
sys.path.insert(0,str(PROJECT_ROOT/'src'))
print(PROJECT_ROOT)

## 2. Imports and deterministic run configuration

In [ ]:
import numpy as np, pandas as pd
from IPython.display import Image,Markdown,display
from piconewton_v3.model import FluidProperties,HydrodynamicConfig,SensorConfig,V2_ARTERY_CASES,V2_EXPECTED_BLOB_SHA,isotropic_validation
from piconewton_v3.provenance import environment_snapshot,execute_stripped_v2,git_commit_or_unknown,strip_notebook_outputs,validate_v2_blob,write_json
from piconewton_v3.study_io import StudyStore,resolve_study_root
from piconewton_v3.workflow import *
CONFIG_PATH=PROJECT_ROOT/'configs'/f'{PROFILE}.json'; CONFIG=json.loads(CONFIG_PATH.read_text())
if STORAGE_MODE_OVERRIDE: CONFIG['storage']['mode']=STORAGE_MODE_OVERRIDE
OUTPUT_ROOT,STORAGE_MODE=resolve_study_root(mode=CONFIG['storage']['mode'],drive_subdir=CONFIG['storage']['drive_subdir'],local_root=CONFIG['storage']['local_root'])
STORE=StudyStore(OUTPUT_ROOT); STORE.initialize_layout(); REPO_ROOT=PROJECT_ROOT.parent
RUN_ID,RUN_ROOT=STORE.create_run(CONFIG,git_commit_or_unknown(REPO_ROOT),V2_EXPECTED_BLOB_SHA,CONFIG['solver_mode'],CONFIG['random_seed'])
STORE.set_status(RUN_ID,'running'); write_json(RUN_ROOT/'provenance/environment.json',environment_snapshot())
print({'run_id':RUN_ID,'run_root':str(RUN_ROOT),'storage':STORAGE_MODE})

## 3. Frozen `v2` hash and cold-regression guard

In [ ]:
V2_PATH=REPO_ROOT/'picoNewton_v2.ipynb'
if V2_PATH.exists():
    write_json(RUN_ROOT/'provenance/v2_hash_guard.json',validate_v2_blob(V2_PATH))
    stripped=RUN_ROOT/'provenance/picoNewton_v2_stripped.ipynb'; write_json(RUN_ROOT/'provenance/v2_stripping.json',strip_notebook_outputs(V2_PATH,stripped))
    if CONFIG.get('run_v2_cold_regression'): write_json(RUN_ROOT/'provenance/v2_cold_execution.json',execute_stripped_v2(stripped,RUN_ROOT/'provenance/picoNewton_v2_executed.ipynb',timeout_s=CONFIG.get('v2_execution_timeout_s',1800)))
elif PROFILE=='publication' or not DEVELOPMENT_SKIP_V2_HASH: raise FileNotFoundError(V2_PATH)
else: display(Markdown('**Development-only:** v2 hash guard skipped outside the parent repository.'))
display(pd.read_csv(PROJECT_ROOT/'tests/fixtures/reproduction_audit_reference.csv'))

## 4. Sources, licences, units and exact arterial inputs

In [ ]:
SOURCE_MANIFEST=pd.read_csv(PROJECT_ROOT/'data/source_manifest.csv'); UNITS=pd.read_csv(PROJECT_ROOT/'data/data_dictionary.csv'); ARTERY_RANGES=pd.read_csv(PROJECT_ROOT/'data/physiological_artery_ranges.csv')
V2_INPUTS=pd.read_csv(PROJECT_ROOT/'data/v2_harmonic_inputs.csv')
assert SOURCE_MANIFEST[['identifier','license','role']].notna().all().all()
for case in V2_ARTERY_CASES: assert np.array_equal(V2_INPUTS.loc[V2_INPUTS.artery_id==case.artery_id,'signed_real_coefficient'].to_numpy(),np.asarray(case.harmonic_coefficients))
display(SOURCE_MANIFEST); display(ARTERY_RANGES)

## 5. Verified anisotropic Womersley solver

In [ ]:
HYDRO_CONFIG=HydrodynamicConfig(radial_order=CONFIG['radial_order'],time_points=CONFIG['time_points'],quadrature_nodes=CONFIG['quadrature_nodes'],beta=CONFIG['beta'],gamma=CONFIG['gamma'],delta=CONFIG['delta'],mode=CONFIG['solver_mode'])
FLUID=FluidProperties(); SENSOR=SensorConfig(); ISOTROPIC_VALIDATION=pd.DataFrame(isotropic_validation(radial_order=HYDRO_CONFIG.radial_order))
if not ISOTROPIC_VALIDATION.passed.all(): STORE.set_status(RUN_ID,'failed'); raise RuntimeError('isotropic validation failed')
display(ISOTROPIC_VALIDATION)

## 6. Real-field Lamb calculation and C0–C13 controls

In [ ]:
CONTROL_TABLE,WAVEFORM_TABLE,HYDRODYNAMICS=run_nominal_controls(V2_ARTERY_CASES,HYDRO_CONFIG,SENSOR,FLUID,seed=CONFIG['random_seed'])
HYDRO_SUMMARY=pd.DataFrame([{'artery_id':k,'artery':v['artery_name'],'alpha':v['alpha'],'force_rms_pN':np.sqrt(np.mean(np.asarray(v['force_signed_n'])**2))*1e12,'WSS_rms_Pa':np.sqrt(np.mean(np.asarray(v['wall_shear_pa'])**2)),'backward_residual':v['max_normalized_backward_residual']} for k,v in HYDRODYNAMICS.items()])
display(HYDRO_SUMMARY); display(CONTROL_TABLE.head())

## 7. Runtime numerical verification

In [ ]:
VERIFICATION_TABLE=runtime_verification_dashboard(V2_ARTERY_CASES,HYDRO_CONFIG,HYDRODYNAMICS,CONTROL_TABLE,FLUID)
STORE.write_csv(f'runs/{RUN_ID}/summaries/runtime_verification.csv',VERIFICATION_TABLE); STORE.register_file(RUN_ID,'summaries/runtime_verification.csv','output')
if not VERIFICATION_TABLE.passed.all(): STORE.set_status(RUN_ID,'failed'); raise RuntimeError('numerical verification failed')
display(VERIFICATION_TABLE)

## 8. Mechanosensor model

\(\dot p=k_+(\Psi)(1-p)-k_-(\Psi)p\), with local-detailed-balance rates and \(\Psi_L=F_Ld_L/(k_BT)\), \(\Psi_\tau=\tau_wV_\tau/(k_BT)\). The exact periodic fixed point is used; signed, reversed-direction and magnitude-sensitive hypotheses remain separate.

In [ ]:
display(CONTROL_TABLE.pivot_table(index=['artery_id','artery_name'],columns='control_id',values='rms').reset_index())

## 9. Parametric sensor grid and physiological Sobol coverage

In [ ]:
PARAMETER_GRID=run_parameter_grid(V2_ARTERY_CASES,HYDRODYNAMICS,FLUID,coupling_lengths_m=CONFIG['coupling_lengths_m'],relaxation_times_s=CONFIG['relaxation_times_s'])
SOBOL_SENSOR_DESIGN=generate_sobol_design(CONFIG['sobol_samples'],seed=CONFIG['random_seed'])
PHYSIOLOGICAL_DESIGN=generate_physiological_design(ARTERY_RANGES,V2_ARTERY_CASES,CONFIG['sobol_samples'],seed=CONFIG['random_seed'])
PHYSIOLOGICAL_SUMMARY,PHYSIOLOGICAL_SPECTRA=run_physiological_coverage(PHYSIOLOGICAL_DESIGN,V2_ARTERY_CASES,HYDRO_CONFIG,SENSOR,checkpoint_dir=RUN_ROOT/'checkpoints/physiological_coverage')
display(PHYSIOLOGICAL_SUMMARY.groupby('artery_name').agg(samples=('sample_id','count'),alpha_min=('alpha','min'),alpha_max=('alpha','max'),effect_median=('effect_parallel_vs_wss','median')))

## 10. Held-out WSS surrogate and immutable E1–E8 gates

In [ ]:
SURROGATE_TABLE,SURROGATE_PARAMETERS=fit_wss_surrogate(V2_ARTERY_CASES,HYDRODYNAMICS,FLUID,SENSOR)
GATE_TABLE=evaluate_effect_gates(PARAMETER_GRID,SURROGATE_TABLE,CONFIG['coupling_lengths_m'],CONFIG['relaxation_times_s'],physiological_coverage=PHYSIOLOGICAL_SUMMARY)
DOMINANCE_TABLE=parameter_dominance(PARAMETER_GRID)
display(SURROGATE_TABLE); display(GATE_TABLE); display(DOMINANCE_TABLE)

## 11. Eight publication figures

In [ ]:
FIGURE_PATHS=generate_publication_figures(RUN_ROOT/'figures',WAVEFORM_TABLE,CONTROL_TABLE,PARAMETER_GRID,SURROGATE_TABLE,GATE_TABLE,DOMINANCE_TABLE)
for p in FIGURE_PATHS: STORE.register_file(RUN_ID,f'figures/{p.name}','output')
for p in FIGURE_PATHS:
    if p.suffix=='.png': display(Image(filename=str(p),width=850))

## 12. Complete dataset, manifests and checksums

In [ ]:
export_publication_dataset(STORE,RUN_ID,waveform_table=WAVEFORM_TABLE,control_table=CONTROL_TABLE,parameter_grid=PARAMETER_GRID,surrogate_table=SURROGATE_TABLE,gate_table=GATE_TABLE,dominance_table=DOMINANCE_TABLE,hydrodynamics=HYDRODYNAMICS)
for relative,table in {'summaries/sobol_sensor_design.csv':SOBOL_SENSOR_DESIGN,'summaries/physiological_design.csv':PHYSIOLOGICAL_DESIGN,'summaries/physiological_coverage.csv':PHYSIOLOGICAL_SUMMARY,'spectra/physiological_force_spectra.csv':PHYSIOLOGICAL_SPECTRA}.items(): STORE.write_csv(f'runs/{RUN_ID}/{relative}',table); STORE.register_file(RUN_ID,relative,'output')
write_json(RUN_ROOT/'summaries/wss_surrogate_parameters.json',SURROGATE_PARAMETERS); STORE.register_file(RUN_ID,'summaries/wss_surrogate_parameters.json','output')
bundle=RUN_ROOT/'publication_bundle'; bundle.mkdir(exist_ok=True)
for src in [PROJECT_ROOT/'data/data_dictionary.csv',PROJECT_ROOT/'data/source_manifest.csv',PROJECT_ROOT/'data/v2_harmonic_inputs.csv',PROJECT_ROOT/'data/publication_dataset_schema.json',PROJECT_ROOT/'configs/effect_gates.json',PROJECT_ROOT/'configs/numerical_thresholds.json',PROJECT_ROOT/'docs/DATA_AVAILABILITY.md',PROJECT_ROOT/'docs/CODE_AVAILABILITY.md',CONFIG_PATH]: shutil.copy2(src,bundle/src.name)
CHECKSUM_PATH=STORE.write_checksums(f'runs/{RUN_ID}','provenance/checksums.sha256'); STORE.set_status(RUN_ID,'complete'); print(RUN_ROOT,CHECKSUM_PATH)

## 13. Availability, limitations and claim selection

In [ ]:
CLAIM_ACTIONS=GATE_TABLE.assign(manuscript_action=np.where(GATE_TABLE.passed,'candidate claim retained','claim removed or explicitly qualified'))
display(CLAIM_ACTIONS)
display(Markdown((PROJECT_ROOT/'docs/DATA_AVAILABILITY.md').read_text())); display(Markdown((PROJECT_ROOT/'docs/CODE_AVAILABILITY.md').read_text()))